# Review support — hand-added labeled addresses (tracker A12)

`data/20241214_labeled_addresses/readme.txt` adds 44 addresses to the labeled set by hand. Labeling a
funder changes the features of every wallet it funded, so reviewers can ask how they were chosen.
The authors' recollection: high-volume addresses (for example Coinbase hot wallets) that rose to the top
of features like provider fan-out, confirmed on Etherscan.

This notebook reconstructs the selection from repository data only:
1. Which of the 44 affect the dataset at all.
2. Whether a label-free ranking (interactor fan-out, network fan-out, tree size) explains the selection,
   on the current and the archived provision network.
3. Which high-ranked funders were *not* added (exceptions a label-free rule would have to explain).
4. A lookup sheet (`hand_labeled_addresses.csv`) with Etherscan links and an empty tag column for manual
   verification. This environment cannot reach Etherscan.

Sybil rates of funded wallets are reported only as an audit diagnostic; they are not a selection criterion.

In [1]:
import os, re, sys
import numpy as np, pandas as pd
sys.path.insert(0, '..')
import sybil_pipeline as sp
pd.set_option('display.width', 250)
DATA = '../data'
P = sp.data_paths(DATA)
ARCHIVE = os.path.join(DATA, '20241114_gas_provision', 'archive', '20241114_layerzero_provisions_000000000000.csv')
readme = open(os.path.join(DATA, '20241214_labeled_addresses', 'readme.txt')).read()
HAND = list(dict.fromkeys(a.lower() for a in re.findall(r"labeled_addresses\.add\('(0x[0-9a-fA-F]{40})'", readme)))
labfile = set(l.strip() for l in open(P['labeled']))
OTHER_LABELED = (labfile - set(HAND)) | {sp.EXTRA_LABELED}      # public label sources only
df = sp.load_l0(P['l0'])
syb = set(pd.read_csv(P['sybil'])['Address'].str.lower())
INTER = set(df['addr'])
print(f'Hand-added: {len(HAND)} ({len(set(HAND) & labfile)} present in the labeled file)')

Hand-added: 44 (44 present in the labeled file)


In [2]:
def profile(path, tag):
    f = pd.read_csv(path, usecols=['activated_address', 'gas_provider'], na_values=['', 'null'])
    f = f[f.gas_provider.notna()].drop_duplicates(['activated_address', 'gas_provider'])
    f['inter'] = f.activated_address.isin(INTER)
    f['sybil'] = f.activated_address.isin(syb) & f.inter
    g = f.groupby('gas_provider').agg(net_fanout=('activated_address', 'size'), int_fanout=('inter', 'sum'),
                                      sybil_funded=('sybil', 'sum'))
    g = g[~g.index.isin(OTHER_LABELED)].copy()            # candidates: not labeled by a public source
    # unlabeled forest (public labels only): subtree size under each candidate
    ok = f[~f.gas_provider.isin(OTHER_LABELED) & ~f.activated_address.isin(OTHER_LABELED)
           & (f.gas_provider != f.activated_address)]
    ch = {}
    for a, p in zip(ok.activated_address, ok.gas_provider):
        ch.setdefault(p, []).append(a)
    def subtree(r):
        seen, st = set(), [r]
        while st:
            x = st.pop()
            if x not in seen:
                seen.add(x); st += ch.get(x, [])
        return len(seen)
    top = g.sort_values('net_fanout', ascending=False).head(3000).index.union(pd.Index(HAND).intersection(g.index))
    g['subtree'] = np.nan
    g.loc[top, 'subtree'] = [subtree(a) for a in top]
    for c in ['int_fanout', 'net_fanout', 'subtree']:
        g[f'rank_{c}'] = g[c].rank(ascending=False, method='min')
    g['hand'] = g.index.isin(HAND)
    g['in_network'] = True
    acts = set(f.activated_address)
    return g.add_prefix(f'{tag}_'), acts

cur, acts_cur = profile(P['gas_prov'], 'cur')
arc, acts_arc = profile(ARCHIVE, 'arc')
t = pd.DataFrame(index=pd.Index(HAND, name='addr'))
t['list_position'] = range(1, len(HAND) + 1)
t['is_interactor'] = t.index.isin(INTER)
t['funds_anyone_cur'] = t.index.isin(cur.index)
t['funds_anyone_arc'] = t.index.isin(arc.index)
t['activated_in_network'] = t.index.isin(acts_cur | acts_arc)
t = t.join(cur.drop(columns=['cur_hand', 'cur_in_network'])).join(arc.drop(columns=['arc_hand', 'arc_in_network']))
effective = t.funds_anyone_cur | t.funds_anyone_arc | t.is_interactor
print(f'Hand-added addresses that fund anyone in either network, or are interactors: {effective.sum()} of {len(t)}')
print(f'No role in either network (labeling them changes nothing): {(~effective).sum()}')
t[['list_position', 'is_interactor', 'funds_anyone_cur', 'cur_int_fanout', 'cur_rank_int_fanout',
   'cur_net_fanout', 'cur_rank_net_fanout', 'arc_subtree', 'arc_rank_subtree']]

Hand-added addresses that fund anyone in either network, or are interactors: 27 of 44
No role in either network (labeling them changes nothing): 17


,list_position,is_interactor,funds_anyone_cur,cur_int_fanout,cur_rank_int_fanout,cur_net_fanout,cur_rank_net_fanout,arc_subtree,arc_rank_subtree
addr,,,,,,,,,
0x00000000000007736e2f9aa5630b8c812e1f3fc9,1,False,True,14.0,220.0,14.0,387.0,15.0,1069.0
0xe93685f3bba03016f02bd1828badd6195988d950,2,True,True,0.0,105235.0,2.0,4473.0,1.0,3000.0
0xd64791e747188b0e5061fc65b56bf20fee2e3321,3,False,False,NaN,NaN,NaN,NaN,NaN,NaN
0x000f422887ea7d370ff31173fd3b46c8f66a5b1c,4,False,True,350.0,6.0,487.0,5.0,766.0,73.0
0x912fd21d7a69678227fe6d08c64222db41477ba0,5,False,True,954.0,2.0,1405.0,2.0,2789.0,12.0
0x230a1ac45690b9ae1176389434610b9526d2f21b,6,False,False,NaN,NaN,NaN,NaN,NaN,NaN
0x151b381058f91cf871e7ea1ee83c45326f61e96d,7,False,True,561.0,4.0,732.0,3.0,988.0,50.0
0x3b794929566e3ba0f25e4263e1987828b5c87161,8,False,True,300.0,7.0,397.0,7.0,588.0,105.0
0x91962711a4d2e4a830b366ce7276d99001e8564b,9,False,True,186.0,11.0,283.0,10.0,517.0,117.0


## Does a label-free ranking explain the selection?

For each ranking: how many of the effective hand-added addresses sit in its top N, and how many top-N funders were *not* added.

In [3]:
rows = []
for tag, g in [('cur', cur), ('arc', arc)]:
    for c in ['int_fanout', 'net_fanout', 'subtree']:
        r = g[f'{tag}_rank_{c}']
        for n in [10, 25, 50, 100]:
            topn = r[r <= n].index
            rows.append(dict(network=tag, ranking=c, top_n=n, hand_in_top=int(pd.Index(HAND).isin(topn).sum()),
                             top_not_added=int((~topn.isin(HAND)).sum())))
cov = pd.DataFrame(rows)
cov.pivot_table(index=['network', 'ranking'], columns='top_n', values=['hand_in_top', 'top_not_added'])

hand_in_top                   top_not_added                  
top_n                      10    25    50    100           10    25    50    100
network ranking                                                                 
arc     int_fanout         5.0   8.0  10.0  13.0           5.0  17.0  42.0  87.0
        net_fanout         6.0  13.0  14.0  18.0           4.0  12.0  36.0  82.0
        subtree            4.0   8.0  10.0  13.0           6.0  17.0  40.0  87.0
cur     int_fanout         5.0   8.0  10.0  13.0           5.0  17.0  42.0  87.0
        net_fanout         6.0  13.0  16.0  18.0           4.0  12.0  34.0  84.0
        subtree            4.0   8.0  12.0  16.0           6.0  17.0  38.0  84.0

## Exceptions: top funders that were not added

Top 30 candidates by network fan-out on the archived network (the likely working version at the time), with the Sybil share of the interactors they funded. If a label-free rule was used, these need a reason other than their funded wallets' labels.

In [4]:
ex = arc.sort_values('arc_net_fanout', ascending=False).head(30).copy()
ex['sybil_rate_funded'] = (ex.arc_sybil_funded / ex.arc_int_fanout.replace(0, np.nan)).round(3)
ex[['arc_net_fanout', 'arc_int_fanout', 'arc_subtree', 'arc_hand', 'sybil_rate_funded']]

,arc_net_fanout,arc_int_fanout,arc_subtree,arc_hand,sybil_rate_funded
gas_provider,,,,,
0xf70da97812cb96acdf810712aa562db8dfa3dbef,2046,2046,2047.0,False,0.000
0x912fd21d7a69678227fe6d08c64222db41477ba0,1411,956,2789.0,True,0.004
0x151b381058f91cf871e7ea1ee83c45326f61e96d,735,561,988.0,True,0.002
0x2fc617e933a52713247ce25730f6695920b3befe,679,656,726.0,False,0.050
0x469503159ddf6bfd0a9ec8eba8e97a84fd3eae5b,500,7,2724.0,True,0.000
0x000f422887ea7d370ff31173fd3b46c8f66a5b1c,487,350,766.0,True,0.000
0x3b794929566e3ba0f25e4263e1987828b5c87161,400,301,588.0,True,0.000
0x4c9af439b1a6761b8e549d8d226a468a6b2803a8,391,358,487.0,False,0.008
0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,295,25,11236.0,True,0.000


In [5]:
# Audit diagnostic: Sybil share among interactors funded by added vs not-added top funders
top100 = arc[arc.arc_rank_net_fanout <= 100]
for flag, grp in top100.groupby('arc_hand'):
    print(f"{'added' if flag else 'not added'}: {len(grp)} funders, {int(grp.arc_int_fanout.sum()):,} funded interactors, "
          f"Sybil share {grp.arc_sybil_funded.sum() / max(grp.arc_int_fanout.sum(), 1) * 100:.2f}%")

not added: 82 funders, 7,223 funded interactors, Sybil share 1.07%
added: 18 funders, 3,100 funded interactors, Sybil share 0.26%


## Lookup sheet for manual Etherscan verification

In [6]:
sheet = t[['list_position', 'is_interactor', 'funds_anyone_cur', 'funds_anyone_arc',
           'cur_int_fanout', 'cur_net_fanout', 'arc_subtree']].copy()
sheet['affects_features'] = effective
sheet['etherscan_url'] = 'https://etherscan.io/address/' + sheet.index
sheet['etherscan_name_tag'] = ''        # fill by hand
sheet['is_contract'] = ''               # fill by hand
sheet['keep_label'] = ''                # yes / no, with reason
sheet.to_csv('hand_labeled_addresses.csv')
ex.assign(etherscan_url='https://etherscan.io/address/' + ex.index, etherscan_name_tag='', keep_label='') \
  .to_csv('top_funders_not_added.csv')
print('Wrote hand_labeled_addresses.csv and top_funders_not_added.csv')

Wrote hand_labeled_addresses.csv and top_funders_not_added.csv
